# TabPFN on the test set: POST-HOC

**What this is:** the team's one-time test (Nov 24) was signed off *without* TabPFN, which needs a GPU. This notebook answers "how would it have done?" after the fact. It **does not change** the reported results or the headline model (LASSO).

**What it does:** fits TabPFN with its frozen settings (no tuning) on 2005–2014 entries and scores the 2019–2020 test cohort. LASSO is scored in the same run for a fair comparison, since library versions shift numbers slightly between machines. Results go to `reports/posthoc_log.md`, kept separate from the signed-off `reports/final_run_log.md`.

**Setup:** Runtime → Change runtime type → **T4 GPU**. Secrets (🔑): `GITHUB_TOKEN` and `TABPFN_TOKEN`, as before. Then Runtime → Run all (about 5–10 minutes).

**Afterwards:** File → Download → Download .ipynb, saved as `notebooks/tabpfn_posthoc_ran.ipynb`.

In [1]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun"

GPU 0: Tesla T4 (UUID: GPU-2e64f31d-1142-4b09-f563-4ada5f72767b)


In [2]:
from google.colab import userdata
import os, subprocess
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/queue-risk-ai"):
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/devault-settings/queue-risk-ai.git", "/content/queue-risk-ai"], check=True)
%cd /content/queue-risk-ai
!git pull -q && git log --oneline -1

/content/queue-risk-ai
b47606e (HEAD -> main, origin/main, origin/HEAD) Swap in the one-page README; point the demo's checkpoints at the frozen spec


In [3]:
!pip install -q -e . tabpfn
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
import tabpfn, torch; print("tabpfn", tabpfn.__version__, "| CUDA:", torch.cuda.is_available())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for queue_risk (pyproject.toml) ... done
tabpfn 10.2.0 | CUDA: True


## Post-hoc test scores

Compare TabPFN with the **LASSO line printed here**, not with the laptop's 0.613, because the environments differ.

In [4]:
!python -W ignore -m queue_risk.posthoc --models "TabPFN,LASSO,Gradient"

POST-HOC (not part of the signed-off test): train 4,020, test 3,181 (3.7% built)
  LASSO logistic: test AUC 0.566 [0.516, 0.621] | out-of-time 2015-18 0.669 | top 20% built 5.8% (1.57x average)
  Gradient boosting: test AUC 0.531 [0.473, 0.587] | out-of-time 2015-18 0.641 | top 20% built 5.2% (1.40x average)

tabpfn-v3.5-20260909.safetensors: downloading bytes:  42% 372M/876M [00:02<00:02, 181MB/s, 31.9MB/s  ]
tabpfn-v3.5-20260909.safetensors: downloading bytes:  57% 503M/876M [00:03<00:01, 211MB/s, 40.6MB/s  ]
tabpfn-v3.5-20260909.safetensors: downloading bytes:  92% 806M/876M [00:04<00:00, 244MB/s, 61.6MB/s  ]
tabpfn-v3.5-20260909.safetensors: reconstructing file:  61% 536M/876M [00:09<00:06, 52.6MB/s, 29.5MB/s  ]
tabpfn-v3.5-20260909.safetensors: downloading bytes: 100% 830M/830M [00:13<00:00, 60.7MB/s, 65.6MB/s  ]
tabpfn-v3.5-20260909.safetensors: reconstructing file: 100% 876M/876M [00:13<00:00, 64.1MB/s, 51.2MB/s  ]
config.json: 100% 36.0/36.0 [00:00<00:00, 116kB/s]
  TabPFN (pre

In [5]:
print(open("reports/posthoc_log.md").read())

# Post-hoc test scores

After the signed-off test run of Nov 24 (reports/final_run_log.md). These do NOT change the reported results or the headline model.

## 2026-11-24 21:59 | commit b47606e | environment: GPU

- LASSO logistic: test AUC 0.566 [0.516, 0.621] | out-of-time 2015-18 0.669 | top 20% built 5.8% (1.57x average)
- Gradient boosting: test AUC 0.531 [0.473, 0.587] | out-of-time 2015-18 0.641 | top 20% built 5.2% (1.40x average)
- TabPFN (pre-trained transformer): test AUC 0.530 [0.471, 0.589] | out-of-time 2015-18 0.645 | top 20% built 4.7% (1.27x average)

